# 🐍 Master Guide: Memory Management, Reference Counting & Cyclic GC
> **Single-Source Revision Guide**: Covers CPython heap allocation (PyMalloc, Arenas, Pools, Blocks), Reference Counting, Cyclic Garbage Collector (Generations 0, 1, 2), and weak references.
>
> *Run each cell interactively with `Shift + Enter`.*

---

## 📌 1. Topic Overview
CPython manages memory automatically via two complementary systems: **Reference Counting** (for instantaneous deallocation) and the **Cyclic Garbage Collector** (for breaking reference cycles).

---
## 🧠 2. CPython Architecture: The PyMalloc Memory Hierarchy

```
┌────────────────────────────────────────────────────────┐
| Layer 3: Object-Specific Allocators (e.g., small ints) |
├────────────────────────────────────────────────────────┤
| Layer 2: PyMalloc (Allocates small objects <= 512 bytes)|
|          - Arena (256 KB memory chunk from OS)         |
|          - Pool  (4 KB memory page inside Arena)       |
|          - Block (Fixed 8-byte increments: 8, 16...512)|
├────────────────────────────────────────────────────────┤
| Layer 1: Python Core Allocator (wraps C malloc/free)   |
├────────────────────────────────────────────────────────┤
| Layer 0: OS Virtual Memory Allocator                   |
└────────────────────────────────────────────────────────┘
```

### The Generational Garbage Collector
Cyclic references (e.g. `a.ref = b; b.ref = a`) cannot be freed by reference counting alone. CPython's GC categorizes objects into **3 generations** based on survival time:
- **Gen 0**: Newly allocated objects. Collected most frequently.
- **Gen 1**: Objects surviving Gen 0 collection.
- **Gen 2**: Long-lived objects surviving multiple collections. Collected least frequently.

---
## 🟢 3. Part 1: Beginner Fundamentals

### Step 1: Inspecting Reference Counts with `sys.getrefcount`

In [ ]:
import sys

a = ["large", "dataset"]
# Note: getrefcount() returns 1 higher than expected because passing 'a' into the function creates an extra reference!
print("Initial ref count of a:", sys.getrefcount(a) - 1)

b = a  # Create reference
print("After b = a            :", sys.getrefcount(a) - 1)

del b  # Remove reference
print("After del b            :", sys.getrefcount(a) - 1)

---
## 🟡 4. Part 2: Intermediate Features

### Breaking Cycles: The `gc` Module & `weakref`
A `weakref` references an object without increasing its `ob_refcnt`. If all strong references are removed, the object is destroyed immediately.

In [ ]:
import weakref
import gc

class CacheTarget:
    def __init__(self, name):
        self.name = name

obj = CacheTarget("ExpensiveModel")
# Create weak reference:
r = weakref.ref(obj)

print("Weak reference alive?", r() is not None)
del obj  # Drop only strong reference
print("After del obj, weak ref:", r())
print("Object was destroyed immediately without needing cyclic GC!")

---
## 📋 5. Part 3: Quick Revision Cheat Sheet

```
+------------------------------------+---------------------------------------+
| Concept                            | Key Rule                              |
+------------------------------------+---------------------------------------+
| Reference counting                 | Primary system; frees memory when cnt=0|
| Cyclic GC                          | Catches self-referencing cycles       |
| 3 Generations                      | Gen 0 (young) -> Gen 1 -> Gen 2 (old) |
| PyMalloc                           | Custom C allocator for objects <= 512B|
| weakref.ref(obj)                   | References without bumping ref count  |
+------------------------------------+---------------------------------------+
```

---
## 🎓 Practical Hands-On & Real-World Examples

### 1. Reference Counting & Cyclic Garbage Collection (`GCEX1.py`, `GCEX2.py`)
> **CPython Memory Architecture**:
> - Primary mechanism: **Reference Counting**. When ref count reaches zero, memory is freed immediately.
> - Secondary mechanism: **Cyclic Garbage Collector** (`gc` module) handles circular self-referencing graphs.
> - `sys.getrefcount(obj)` returns current active references (+1 for the function call itself).

In [ ]:
# CPython Memory Management & Reference Tracking Drill
import sys
import gc

# 1. Reference counting lifecycle
sample_list = [1, 2, 3]
print("Initial ref count: ", sys.getrefcount(sample_list))  # Typically 2 (object + arg to getrefcount)

alias_ref = sample_list
print("After second alias:", sys.getrefcount(sample_list))  # 3

del alias_ref
print("After deleting alias:", sys.getrefcount(sample_list))  # Back to 2

# 2. Cyclic reference graph and gc.collect()
class Node:
    def __init__(self, name):
        self.name = name
        self.partner = None

node_a = Node("A")
node_b = Node("B")
node_a.partner = node_b
node_b.partner = node_a  # Circular reference!

# Break external pointers
del node_a
del node_b

# Collect cyclic garbage
unreachable_count = gc.collect()
print(f"Cyclic objects collected by gc.collect(): {unreachable_count}")